# Real vLLM inference with TRITON_ATTN

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet on only if the model must be downloaded. Run notebooks in order 01 → 06; each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Loads `Qwen/Qwen2.5-0.5B-Instruct` (public, ~1 GB fp16) from Hugging Face, runs short-decode and ~1.2k-token prefill generations at TP=1 and TP=2, checks determinism, and **greps vLLM's own log** for the selected backend. Set `HF_TOKEN` as a Kaggle secret only for gated models.

In [ ]:
import glob, os, pathlib
# Locate the release bundle (the Kaggle dataset holding manifests/SHA256SUMS). Override with STOKASTISK_BUNDLE.
hits = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
BUNDLE = pathlib.Path(os.environ.get("STOKASTISK_BUNDLE") or (hits[0] if hits else "")) 
assert (BUNDLE / "manifests" / "SHA256SUMS").is_file(), "attach the stokastisk-0.3.0 release dataset to this notebook"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("bundle:", BUNDLE)

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""), VLLM_LOGGING_LEVEL="INFO")
for tp in (1, 2):
    with open(f"/kaggle/working/stokastisk-acceptance/vllm_tp{tp}.log", "w") as f:
        rc = subprocess.run([sys.executable, str(SCRIPTS / "05_inference.py"), str(tp), "Qwen/Qwen2.5-0.5B-Instruct"],
                            env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    print("TP", tp, "rc", rc)

In [ ]:
import re, json
OUT = "/kaggle/working/stokastisk-acceptance"
ev = {}
for tp in (1, 2):
    log = open(f"{OUT}/vllm_tp{tp}.log", errors="replace").read()
    chosen = re.findall(r"Using (?:AttentionBackendEnum\.)?TRITON_ATTN backend", log)
    other = re.findall(r"(?i)using[^\n]*(?:FLASH_?ATTN|FLASHINFER|FLASHMLA|FLEX_ATTENTION|XFORMERS)[^\n]*backend", log)
    ev[tp] = {"triton_lines": len(chosen), "other_backend_lines": other[:3], "log_bytes": len(log)}
    print(f"TP{tp}: {ev[tp]}")
    assert len(chosen) >= tp, f"expected >= {tp} 'Using ...TRITON_ATTN backend' log line(s) (one per worker); inspect {OUT}/vllm_tp{tp}.log"
    assert not other, "a non-Triton attention backend was reported"
    assert "Selected backend" not in log or "is not valid" not in log, "vLLM rejected the selected backend"
json.dump(ev, open(f"{OUT}/05_backend_log_evidence.json", "w"), indent=1)

In [ ]:
print(open("/kaggle/working/stokastisk-acceptance/05_inference_tp1.json").read()[:1500])